In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Capa Bronce — TFM TUI

**Objetivo:** ingesta de los datos **crudos** (raw), tal cual llegan, sin transformar.
Solo se hace lo mínimo para que la tabla sea legible: saneo de nombres de columna
(BOM / espacios) y una columna de trazabilidad `_fichero_origen`.

Limpieza, KPIs, conversión de coordenadas y cruces entre tablas → capa **Plata**.

Tablas que genera:
- `bronze.Familia_POI` — puntos de interés (parques, museos, monumentos...)
- `bronze.Familia_Restaurantes` / `bronze.Familia_Restaurantes_terrazas` — censo de locales
- `bronze.Familia_Paradas` — paradas de transporte público (GTFS CRTM)


## 1. Setup — ruta, esquema y funciones base

In [2]:
!pip install pyspark delta-spark

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder.appName("TFM_Bronze_Layer") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.0/55.0 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: importlib_metadata
    Found existing installation: importlib_metadata 9.0.0
    Uninstalling importlib_metadata-9.0.0:
      Successfully uninstalled importlib_metadata-9.0.0


In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd

# --- AJUSTA ESTAS DOS RUTAS a como se llamen vuestras carpetas en Drive ---
ROOT   = "/content/drive/MyDrive/Master/TFM TUI 3/Raw"      # carpeta con los CSV crudos
BRONCE = "/content/drive/MyDrive/Master/TFM TUI 3/Bronce"   # salida (equivale al esquema bronze)
os.makedirs(BRONCE, exist_ok=True)


def leer_csv(rel_path, encoding="ISO-8859-1", sep=";"):
    """Lee un CSV crudo y sanea los nombres de columna.
    encoding: la mayoria son ISO-8859-1; censo y paradas son UTF-8.
    sep: ';' en los ficheros de Madrid; ',' en el GTFS de paradas."""
    df = pd.read_csv(f"{ROOT}/{rel_path}", sep=sep, encoding=encoding,
                     dtype=str, quotechar='"')          # dtype=str -> raw, sin inferir tipos
    df.columns = [c.replace("\ufeff", "").replace('"', "").strip() for c in df.columns]
    df["_fichero_origen"] = rel_path
    return df


def guardar(df, tabla):
    """Guarda un DataFrame como Parquet en la carpeta bronce."""
    df.to_parquet(f"{BRONCE}/{tabla}.parquet", index=False)
    print(f"OK  {tabla}  ->  {len(df)} filas, {df.shape[1]} columnas")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. (Opcional) Explorar los ficheros disponibles

In [4]:
for dirpath, _, filenames in os.walk(ROOT):
    nivel = dirpath.replace(ROOT, "").count(os.sep)
    print("  " * nivel + os.path.basename(dirpath) + "/")
    for f in sorted(filenames):
        if f.lower().endswith(".csv"):
            print("  " * (nivel + 1) + f)


Raw/
  unificado_stops.csv
  M-Actividades culturales/
    206974-4-agenda-eventos-culturales-100-csv.csv
  M-Transporte/
    EMT/
      demandadialinea.csv
      linesemt.csv
    Taxi/
      208094-4-reserva-paradas-taxis-csv.csv
    Metro/
    Bicimad/
      205062-0-reservas-moto-csv.csv
      205099-2-aparca-bicis.csv
      historicousuariosactivos.csv
    Trafico/
  M-Parques/
    200761-0-parques-jardines-csv.csv
    300153-18-zonas-verdes-inventario.csv
  M-Deporte y ocio/
    200215-0-instalaciones-deportivas-csv.csv
    200637-3-areas-mayores.csv
    200652-0-areas-infantiles.csv
    201132-1-museos-csv.csv
    208844-5-monumentos-edificios-csv.csv
    208862-11-ocio_salas-csv.csv
    210227-0-piscinas-publicas-csv.csv
    300390-0-areas-deportivas.csv
  Opiniones/
  CM-Transporte/
    demanda-del-servicio-publico-de-bicicletas-por-municipios.csv
    ocupacion-media-de-los-vehiculos-por-medio-de-transporte.csv
    ratios-del-metro-de-madrid.csv
    viajeros-subidos-y-bajados-e

## 3. Familia A — Puntos de interés (POI)

Ficheros de puntos de interés de Madrid apilados en una sola tabla, con la
columna `categoria` para distinguir el tipo de cada fila.

In [5]:
FAMILIA_A = {
    "M-Parques/200761-0-parques-jardines-csv.csv":                "parque",
    "M-Deporte y ocio/200215-0-instalaciones-deportivas-csv.csv": "instalacion_deportiva",
    "M-Deporte y ocio/201132-1-museos-csv.csv":                   "museo",
    "M-Deporte y ocio/208844-5-monumentos-edificios-csv.csv":     "monumento",
    "M-Deporte y ocio/208862-11-ocio_salas-csv.csv":              "sala_ocio",
    "M-Deporte y ocio/210227-0-piscinas-publicas-csv.csv":        "piscina",
    "M-Restaurantes y mercados/200967-4-mercados-csv.csv":        "mercado",
    "M-Restaurantes y mercados/202105-1-mercadillos-csv.csv":     "mercadillo",
}

dfs = []
for path, cat in FAMILIA_A.items():
    df = leer_csv(path)
    df["categoria"] = cat
    dfs.append(df)

familia_a = pd.concat(dfs, ignore_index=True)   # equivale a unionByName(allowMissingColumns=True)

guardar(familia_a, "Familia_POI")
display(familia_a["categoria"].value_counts().sort_index())


OK  Familia_POI  ->  1402 filas, 35 columnas


,count
categoria,
instalacion_deportiva,607
mercadillo,33
mercado,45
monumento,346
museo,68
parque,208
piscina,56
sala_ocio,39


## 4. Censo de locales (crudo)

- `-5` → **locales** (uno por local)
- `-6` → **terrazas** (varias por local, con superficies)

Se ingieren tal cual. El cruce locales↔terrazas y la conversión de coordenadas
UTM → lat/lon se harán en la capa **Plata**.

In [6]:
censo_locales  = leer_csv("M-Restaurantes y mercados/200085-5-censo-locales.csv", encoding="UTF-8")
censo_terrazas = leer_csv("M-Restaurantes y mercados/200085-6-censo-locales.csv", encoding="UTF-8")

guardar(censo_locales,  "Familia_Restaurantes")
guardar(censo_terrazas, "Familia_Restaurantes_terrazas")

OK  Familia_Restaurantes  ->  225375 filas, 48 columnas
OK  Familia_Restaurantes_terrazas  ->  6583 filas, 118 columnas


## 5. Paradas de transporte público (GTFS CRTM)

Fichero GTFS del Consorcio de Transportes: paradas de metro, cercanías y autobús.
Distinto formato al resto: **separador coma** y **encoding UTF-8**.
Ya trae `stop_lat` / `stop_lon` en WGS84, así que **no hay que convertir coordenadas**.

> El fichero está en la raíz del volumen (`tfm_tui_data/`).

In [7]:
paradas_transporte = leer_csv("unificado_stops.csv", encoding="UTF-8", sep=",")

guardar(paradas_transporte, "Familia_Paradas")
display(paradas_transporte["tipo_transporte"].value_counts())

OK  Familia_Paradas  ->  9553 filas, 15 columnas


,count
tipo_transporte,
Ferrocarril Cercanías / Red de Metro,8406
Autobuses Interurbanos,1050
Autobuses Urbanos (Otros Municipios / EMT),97


## 6. Comprobación rápida

In [8]:
for t in ["Familia_POI", "Familia_Restaurantes", "Familia_Restaurantes_terrazas", "Familia_Paradas"]:
    df = pd.read_parquet(f"{BRONCE}/{t}.parquet")
    print(f"--- {t} ---")
    print(df.dtypes)
    print()


--- Familia_POI ---
PK                     object
NOMBRE                 object
DESCRIPCION-ENTIDAD    object
HORARIO                object
EQUIPAMIENTO           object
TRANSPORTE             object
DESCRIPCION            object
ACCESIBILIDAD          object
CONTENT-URL            object
NOMBRE-VIA             object
CLASE-VIAL             object
TIPO-NUM               object
NUM                    object
PLANTA                 object
PUERTA                 object
ESCALERAS              object
ORIENTACION            object
LOCALIDAD              object
PROVINCIA              object
CODIGO-POSTAL          object
COD-BARRIO             object
BARRIO                 object
COD-DISTRITO           object
DISTRITO               object
COORDENADA-X           object
COORDENADA-Y           object
LATITUD                object
LONGITUD               object
TELEFONO               object
FAX                    object
EMAIL                  object
TIPO                   object
_fichero_origen     